In [3]:
!pip install -q langchain langchain-groq langchain-core python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00


In [5]:
# =========================
# Imports and LLM Setup
# =========================

import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Set your Groq API key
os.environ["GROQ_API_KEY"] = "Enter the Groq API key: "

# Router model (deterministic)
router_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# Expert model (slightly more flexible)
expert_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

parser = StrOutputParser()

In [6]:
# =========================
# Router Prompt
# =========================

router_prompt = ChatPromptTemplate.from_template("""
You are a query routing assistant.

Classify the user query into ONLY one of these categories:
- Technical
- Billing
- General

Rules:
- Technical: coding, programming, debugging, software, APIs, errors, development issues
- Billing: payment, subscription, invoice, refund, charges, pricing
- General: anything else

Return ONLY the category name.
Query: {query}
""")

router_chain = router_prompt | router_llm | parser

In [7]:
# =========================
# Expert Handlers
# =========================

def technical_expert(query):
    return f"Technical Expert Response: This looks like a technical issue.\nQuery: {query}"

def billing_expert(query):
    return f"Billing Expert Response: This appears to be a billing-related concern.\nQuery: {query}"

def general_expert(query):
    return f"General Expert Response: This is a general-purpose query.\nQuery: {query}"

In [8]:
# =========================
# Router Function
# =========================

def route_query(user_query):
    result = router_chain.invoke({"query": user_query}).strip()

    if "Technical" in result:
        return "Technical"
    elif "Billing" in result:
        return "Billing"
    else:
        return "General"

In [9]:
# =========================
# Main Dispatcher
# =========================

def handle_query(user_query):
    route = route_query(user_query)

    print("User Query:", user_query)
    print("Predicted Route:", route)
    print("-" * 50)

    if route == "Technical":
        return technical_expert(user_query)
    elif route == "Billing":
        return billing_expert(user_query)
    else:
        return general_expert(user_query)

In [10]:
# =========================
# Router Test 1: Technical
# =========================

technical_query = "How do I fix a syntax error in Python?"
technical_output = handle_query(technical_query)
print(technical_output)

User Query: How do I fix a syntax error in Python?
Predicted Route: Technical
--------------------------------------------------
Technical Expert Response: This looks like a technical issue.
Query: How do I fix a syntax error in Python?


In [11]:
# =========================
# Temperature Check
# =========================

print("Router Temperature:", 0)
print("Expert Temperature:", 0.7)

Router Temperature: 0
Expert Temperature: 0.7


In [12]:
# =========================
# Bonus Challenge: Tool Use Expert
# =========================

def get_mock_crypto_price(asset_name):
    mock_prices = {
        "bitcoin": "$68,450",
        "ethereum": "$3,420"
    }
    return mock_prices.get(asset_name.lower(), "Price not available")

def tool_use_expert(user_query):
    q = user_query.lower()

    if "bitcoin" in q and ("price" in q or "current" in q):
        price = get_mock_crypto_price("bitcoin")
        return f"Tool Response: The current price of Bitcoin is {price} (mock fetched data)."

    return "No tool needed."

def enhanced_route_query(user_query):
    q = user_query.lower()

    if "bitcoin" in q or "crypto" in q or "price" in q:
        return "Tool Use Expert"
    return route_query(user_query)

# Test Tool Use Expert
tool_query = "What is the current price of Bitcoin?"
selected_route = enhanced_route_query(tool_query)

print("User Query:", tool_query)
print("Predicted Route:", selected_route)

if selected_route == "Tool Use Expert":
    print(tool_use_expert(tool_query))
else:
    print(handle_query(tool_query))

User Query: What is the current price of Bitcoin?
Predicted Route: Tool Use Expert
Tool Response: The current price of Bitcoin is $68,450 (mock fetched data).
